In [1]:
# =========================
# 1. Imports
# =========================
import os
import shutil
import random
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings("ignore")
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, Flatten
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.optimizers import Adamax
from tensorflow.keras.applications import EfficientNetB3
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

# =========================
# 2. Dataset Paths
# =========================
brain_tumor_paths = {
    "Normal": "/kaggle/input/multibraindisease/archive (16)/Data/Normal",
    "Glioma": "/kaggle/input/multibraindisease/archive (16)/Data/Tumor/glioma_tumor",
    "Meningioma": "/kaggle/input/multibraindisease/archive (16)/Data/Tumor/meningioma_tumor",
    "Pituitary": "/kaggle/input/multibraindisease/archive (16)/Data/Tumor/pituitary_tumor"
}

alz_paths = {
    "MildDemented": "/kaggle/input/multibraindisease/archive (17)/AugmentedAlzheimerDataset/MildDemented",
    "ModerateDemented": "/kaggle/input/multibraindisease/archive (17)/AugmentedAlzheimerDataset/ModerateDemented",
    "NonDemented": "/kaggle/input/multibraindisease/archive (17)/AugmentedAlzheimerDataset/NonDemented",
    "VeryMildDemented": "/kaggle/input/multibraindisease/archive (17)/AugmentedAlzheimerDataset/VeryMildDemented"
}

ms_paths = {
    "Control_Axial": "/kaggle/input/multibraindisease/archive (18)/MS/Control Axial_crop",
    "Control_Saggital": "/kaggle/input/multibraindisease/archive (18)/MS/Control Saggital_crop",
    "MS_Axial": "/kaggle/input/multibraindisease/archive (18)/MS/MS Axial_crop",
    "MS_Saggital": "/kaggle/input/multibraindisease/archive (18)/MS/MS Saggital_crop"
}

# =========================
# 3. Balance dataset (1000 images per class)
# =========================
combined_root = "/kaggle/working/balanced_combined_dataset"
os.makedirs(combined_root, exist_ok=True)

def balance_and_copy(src_dict, dst_root, target_count=1000):
    for label, src_path in src_dict.items():
        dst_path = os.path.join(dst_root, label)
        os.makedirs(dst_path, exist_ok=True)
        files = os.listdir(src_path)
        random.shuffle(files)

        if len(files) >= target_count:
            selected = files[:target_count]
        else:
            selected = files
            extra = np.random.choice(files, size=target_count-len(files), replace=True)
            selected = list(selected) + list(extra)

        for i, f in enumerate(selected):
            shutil.copy(os.path.join(src_path, f), os.path.join(dst_path, f"{i}_{f}"))

# Balance datasets
balance_and_copy(brain_tumor_paths, combined_root)
balance_and_copy(alz_paths, combined_root)
balance_and_copy(ms_paths, combined_root)

# =========================
# 4. Create DataFrame
# =========================
classes = os.listdir(combined_root)
filepaths, labels = [], []

for cls in classes:
    cls_path = os.path.join(combined_root, cls)
    for f in os.listdir(cls_path):
        filepaths.append(os.path.join(cls_path, f))
        labels.append(cls)

df = pd.DataFrame({"filepaths": filepaths, "labels": labels})
print(df.head())
print(df["labels"].value_counts())

# =========================
# 5. Stratified train/val/test split
# =========================
train_val, test_set = train_test_split(
    df, test_size=0.15, stratify=df['labels'], random_state=42
)
train_set, val_set = train_test_split(
    train_val, test_size=0.1765, stratify=train_val['labels'], random_state=42
)

print("Train:", train_set.shape, "Val:", val_set.shape, "Test:", test_set.shape)

# =========================
# 6. Image Generators
# =========================
image_gen = ImageDataGenerator(preprocessing_function=tf.keras.applications.efficientnet.preprocess_input)

train = image_gen.flow_from_dataframe(
    train_set, x_col="filepaths", y_col="labels",
    target_size=(244,244), color_mode="rgb",
    class_mode="categorical", batch_size=32, shuffle=True
)

val = image_gen.flow_from_dataframe(
    val_set, x_col="filepaths", y_col="labels",
    target_size=(244,244), color_mode="rgb",
    class_mode="categorical", batch_size=32, shuffle=False
)

test = image_gen.flow_from_dataframe(
    test_set, x_col="filepaths", y_col="labels",
    target_size=(244,244), color_mode="rgb",
    class_mode="categorical", batch_size=32, shuffle=False
)

# =========================
# 7. Build EfficientNetB3 model
# =========================
img_shape = (244,244,3)
base_model = EfficientNetB3(include_top=False, weights="imagenet", input_shape=img_shape, pooling="max")
base_model.trainable = True

set_trainable = False
for layer in base_model.layers:
    if layer.name == "block6a_expand_conv":
        set_trainable = True
    layer.trainable = set_trainable

model = Sequential([
    tf.keras.layers.InputLayer(input_shape=img_shape),
    base_model,
    Flatten(),
    Dropout(0.3),
    Dense(128, activation="relu"),
    Dropout(0.25),
    Dense(len(classes), activation="softmax")
])

model.compile(optimizer=Adamax(1e-3), loss="categorical_crossentropy", metrics=["accuracy"])
model.summary()

# =========================
# 8. Train model
# =========================
from tensorflow.keras.callbacks import EarlyStopping, LearningRateScheduler

annealer = LearningRateScheduler(lambda x: 1e-3 * 0.95**x, verbose=1)
early_stop = EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True)

history = model.fit(train, validation_data=val, epochs=10, callbacks=[annealer, early_stop])

# =========================
# 9. Plot accuracy
# =========================
plt.plot(history.history['accuracy'])
plt.plot(history.history['val_accuracy'])
plt.title('Model Accuracy')
plt.ylabel('Accuracy')
plt.xlabel('Epoch')
plt.legend(['Train', 'Val'], loc='upper left')
plt.show()

# =========================
# 10. Save model
# =========================
model.save("EfficientNetB3_combined_dataset.keras")

# =========================
# 11. Evaluate on Test Set
# =========================
test.reset()
preds = model.predict(test, verbose=1)
y_pred = np.argmax(preds, axis=1)
y_true = test.classes
class_names = list(test.class_indices.keys())

print(classification_report(y_true, y_pred, target_names=class_names))

cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(10,8))
sns.heatmap(cm, annot=True, fmt="d", xticklabels=class_names, yticklabels=class_names, cmap="Blues")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix")
plt.show()


2025-11-02 12:01:22.565658: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1762084882.818127      13 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1762084882.884914      13 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


FileNotFoundError: [Errno 2] No such file or directory: '/kaggle/input/multibraindisease/archive (16)/Data/Normal'

In [ ]:
model.save("EfficientNetB3_combined_dataset.h5")

In [ ]:
# =========================
# 9. Plot accuracy & loss
# =========================
plt.figure(figsize=(12,5))

# Accuracy
plt.subplot(1,2,1)
plt.plot(history.history['accuracy'], label='Train Accuracy')
plt.plot(history.history['val_accuracy'], label='Val Accuracy')
plt.title('Model Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()

# Loss
plt.subplot(1,2,2)
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Val Loss')
plt.title('Model Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()

plt.show()


In [ ]:
from graphviz import Digraph
from IPython.display import Image, display

# Create Digraph
dot = Digraph(comment='EfficientNetB3 Brain Model', format='png')
dot.attr(rankdir='TB', size='10')

# Define nodes with colors
dot.node('Input', 'Input Layer\n(244x244x3)', shape='rect', style='filled', color='#FFD700')  # yellow
dot.node('EffB3', 'EfficientNetB3\n(pretrained, 449 layers,\n fine-tuned from block6a_expand_conv)', 
         shape='rect', style='filled', color='#FF6347')  # red
dot.node('Flatten', 'Pooling(Max) -> Flatten\n(1D conversion)', shape='rect', style='filled', color='#90EE90')  # green
dot.node('DO1', 'Dropout\n(rate=0.3)', shape='rect', style='filled', color='#87CEFA')  # light blue
dot.node('Dense1', 'Dense Layer\n128 neurons, ReLU', shape='rect', style='filled', color='#DA70D6')  # purple
dot.node('DO2', 'Dropout\n(rate=0.25)', shape='rect', style='filled', color='#87CEFA')  # light blue
dot.node('Output', f'Output Layer\n({len(classes)} classes, Softmax)', shape='rect', style='filled', color='#D8BFD8')  # lavender

# Connect nodes
dot.edges([
    ('Input', 'EffB3'),
    ('EffB3', 'Flatten'),
    ('Flatten', 'DO1'),
    ('DO1', 'Dense1'),
    ('Dense1', 'DO2'),
    ('DO2', 'Output')
])

# Render to file
file_path = '/kaggle/working/EfficientNetB3_brain_model_diagram.png'
dot.render('/kaggle/working/EfficientNetB3_brain_model_diagram', view=False)

# Display the image in notebook
display(Image(filename=file_path))
